# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is.

The Monitoring page reads two sources, and the functions below (the ones behind
the page) turn them into numbers:
- the **run records**, one per run in its final state: activity, human review,
  tokens and cost, latency and failures;
- the **event log**, for what can happen several times in a run: guardrail
  firings, capability executions and replays.

To make sure there is something to show, this notebook first runs the five example
workflows through the graph in mock mode (approving only 1 and 5, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.graph import run_graph
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    state = run_graph(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, state["status"])

chf_safe_haven_success completed
spread_frequency_mismatch_block blocked
arima_forecast_block blocked
prompt_injection_unknown_capability blocked


buy_the_dip_tech completed


In [3]:
from quantlab.lineage import load_runs
from quantlab.monitoring import activity, monitored_runs, read_events

records = load_runs(settings.runs_dir)
events = read_events(settings.events_path)
counted = monitored_runs(records)
print(len(records), "run records,", len(counted), "counted by the page,", len(events), "events")
pd.Series(activity(records))

345 run records, 121 counted by the page, 3919 events


total                      345.000000
completed                  130.000000
blocked                     67.000000
planner_refused             83.000000
failed                       0.000000
rejected                     5.000000
completed_share              0.376812
blocked_share                0.194203
planner_refused_share        0.240580
failed_share                 0.000000
rejected_share               0.014493
completed_percent           38.000000
blocked_percent             19.000000
planner_refused_percent     24.000000
failed_percent               0.000000
rejected_percent             2.000000
dtype: float64

The page counts only the decided runs started on the Analysis page
(`monitored_runs`, printed above), and replays from every source: these headless runs
are not among them. To show what each function computes, the cells below apply them
to every run record.

`activity` splits the runs into five outcomes that do not overlap (completed, blocked
by a deterministic guardrail, refused by the planner with P01, failed, rejected by the
reviewer), so they add up to the total; `_percent` holds the whole percentages the
page shows, which add up to 100.

In [4]:
from quantlab.monitoring import (
    completed_with_warnings,
    grounded_narratives,
    replay_results,
    review_outcomes,
)

pd.Series(
    {
        **review_outcomes(records),
        **replay_results(events),
        **grounded_narratives(records),
        **completed_with_warnings(records),
    }
)

approved               127.000000
rejected                 5.000000
approval_rate            0.962121
edited                  17.000000
edit_rate                0.133858
approval_median_s        0.037000
approval_p95_s         103.289600
replays                 14.000000
matching                14.000000
replay_match_rate        1.000000
narratives             130.000000
grounded               130.000000
grounded_rate            1.000000
completed              130.000000
with_warnings           56.000000
with_warnings_share      0.430769
dtype: float64

Every guardrail and every capability, zeros included, most frequent first:

In [5]:
from quantlab.monitoring import capability_counts, guardrail_counts

pd.DataFrame(guardrail_counts(events))

,code,name,action,count
0,V10,MATERIAL_ASSUMPTION,APPROVE,549
1,V09,LOOK_AHEAD,APPROVE,97
2,P01,PLANNER_REFUSED,BLOCK,89
3,E04,ALIGNMENT_LOSS,WARN,71
4,V08,FREQUENCY_MISMATCH,BLOCK,47
5,V03,INVALID_PARAM,BLOCK,36
6,V02,UNKNOWN_CAPABILITY,BLOCK,34
7,R01,REQUEST_TOO_LONG,BLOCK,6
8,V05,BUDGET_EXCEEDED,BLOCK,1
9,E05,LOW_EFFECTIVE_SAMPLE,WARN,1


In [6]:
pd.DataFrame(capability_counts(events))

,capability,code,count
0,load_series,C01,188
1,resample,C02,168
2,flag_regime,C07,124
3,hac_mean_difference,C10,95
4,forward_return,C08,93
5,compute_returns,C03,62
6,compare_groups,C09,54
7,rolling_volatility,C06,26
8,spread,C05,19
9,difference,C04,1


Cost per run, from the tokens the API reported and the prices in
`config.LLM_PRICES_PER_MILLION_TOKENS`. Mock runs call no LLM and are left out, so
in mock mode the table has no runs and the cache saved nothing:

In [7]:
from quantlab.config import LLM_PRICES_PER_MILLION_TOKENS
from quantlab.monitoring import cache_savings, cost_table

display(pd.DataFrame(cost_table(records, LLM_PRICES_PER_MILLION_TOKENS)))
pd.Series(cache_savings(records, LLM_PRICES_PER_MILLION_TOKENS))

,stage,runs,input_mean,input_p95,output_mean,output_p95,cost_mean,cost_p95,cost_total
0,planner,116,5298.267241,11642.50,574.258621,1714.25,0.000817,0.001902,0.094767
1,narrator,116,427.000000,1578.00,57.284483,195.00,0.000071,0.000253,0.008276
2,total,116,5725.267241,11988.25,631.543103,1844.25,0.000888,0.002128,0.103043


planner_calls            112.000000
cache_hits                10.000000
hit_rate                   0.089286
saved_input_tokens     47946.000000
saved_output_tokens     5679.000000
saved_cost                 0.007634
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds), and the most recent
failures with the stage and the guardrail that stopped them:

In [8]:
from quantlab.monitoring import latency_table, recent_failures

pd.DataFrame(latency_table(records))

,stage,runs,mean_ms,p95_ms,recorded
0,Request check,0,NaN,NaN,False
1,Planner,339,2537.825959,11730.4600,True
2,Validator,339,0.154749,0.3910,True
3,Executor,130,135.194077,226.0465,True
4,Narrator,130,964.485385,3700.1950,True


In [9]:
pd.DataFrame(recent_failures(records, limit=5))

,time,run_id,status,stage,code,message
0,2026-10-20T11:24:38.664Z,20261020-112438-a1ab10,blocked,Validator,V02,The plan was rejected (V02): write a new request.
1,2026-10-20T11:24:38.630Z,20261020-112438-1b236b,blocked,Planner,P01,The plan has blocking issues: P01:plan
2,2026-10-20T11:24:38.593Z,20261020-112438-86ef16,blocked,Validator,V08,The plan has blocking issues: V08:s4
3,2026-10-20T10:56:44.940Z,20261020-105644-cfd907,blocked,Validator,V02,The plan was rejected (V02): write a new request.
4,2026-10-20T10:56:18.061Z,20261020-105618-2ff830,blocked,Planner,P01,The plan has blocking issues: P01:plan
